# LoRA fine-tuning workflow

The notebook owns data loading, training LoRA, and scoring.

In [1]:
#helpers for LoRA

class LoRA_Comp:
    def __init__(self,L_RANK,L_ALPHA,TARGET_MOD,L_DROPOUT,L_BIAS,T_TYPE):
        self.LORA_RANK = L_RANK
        self.LORA_ALPHA = L_ALPHA
        self.TARGET_MODULES = TARGET_MOD
        self.LORA_DROPOUT =L_DROPOUT
        self.LORA_BIAS =L_BIAS
        self.TASK_TYPE = T_TYPE


def build_lora_config(lora_component, **overrides):
    config = {
        "r": lora_component.LORA_RANK,
        "lora_alpha": lora_component.LORA_ALPHA,
        "target_modules": lora_component.TARGET_MODULES,
        "lora_dropout": lora_component.LORA_DROPOUT,
        "bias": lora_component.LORA_BIAS,
        "task_type": lora_component.TASK_TYPE,
    }
    config.update(overrides)
    return LoraConfig(**config)


def attach_lora(model, lora_component, **overrides):
    lora_config = build_lora_config(lora_component, **overrides)
    return get_peft_model(model, lora_config)

In [2]:
#imports

import site
cutlass_pkg_path = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/"
site.addsitedir(cutlass_pkg_path)# for mamba_ssm

from pathlib import Path
import kagglehub
import mamba_ssm


import polars as pl
import torch

from peft import LoraConfig, get_peft_model
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


In [1]:
# utilities
from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable

import math
import polars as pl
import re
import torch.nn as nn


@dataclass
class ExperimentContext:
    train_csv: Path
    test_csv: Path
    model_path: Path
    train_df: pl.DataFrame
    val_df: pl.DataFrame
    test_df: pl.DataFrame
    score_fn: Callable[[list[str], list[str]], dict[str, Any]]
    config: dict[str, Any] = field(default_factory=dict)


def resolve_model_path(model_path: str | Path) -> Path:
    path = Path(model_path)
    snapshots_dir = path / "snapshots"
    if snapshots_dir.exists():
        snapshots = sorted(p for p in snapshots_dir.iterdir() if p.is_dir())
        if snapshots:
            return snapshots[-1]
    return path


def make_train_val_split(
    train_df: pl.DataFrame,
    max_train_rows: int,
    max_val_rows: int,
) -> tuple[pl.DataFrame, pl.DataFrame]:
    total_needed = max_train_rows + max_val_rows
    small_df = train_df.head(total_needed)
    val_df = small_df.head(max_val_rows)
    fit_df = small_df.slice(max_val_rows, max_train_rows)
    return fit_df, val_df


def extract_final_answer(text: str | None) -> str:
    if text is None:
        return "NOT_FOUND"

    boxed_starts = list(re.finditer(r"\\boxed\{", text))
    matches = []
    for i, match in enumerate(boxed_starts):
        start = match.end()
        end = boxed_starts[i + 1].start() if i + 1 < len(boxed_starts) else len(text)
        segment = text[start:end]
        last_brace = segment.rfind("}")
        matches.append(segment[:last_brace] if last_brace != -1 else segment)
    if matches:
        non_empty = [match.strip() for match in matches if match.strip()]
        if non_empty:
            return non_empty[-1]
        return matches[-1].strip()

    patterns = [
        r"The final answer is:\\s*([^\\n]+)",
        r"Final answer is:\\s*([^\\n]+)",
        r"Final answer\\s*[:：]\\s*([^\\n]+)",
        r"final answer\\s*[:：]\\s*([^\\n]+)",
    ]
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            return matches[-1].strip()

    matches = re.findall(r"-?\\d+(?:\\.\\d+)?", text)
    if matches:
        return matches[-1]

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[-1] if lines else "NOT_FOUND"


def verify_answer(stored_answer: str, predicted: str) -> bool:
    stored_answer = stored_answer.strip()
    predicted = predicted.strip()

    if re.fullmatch(r"[01]+", stored_answer):
        return predicted.lower() == stored_answer.lower()

    try:
        stored_num = float(stored_answer)
        predicted_num = float(predicted)
        return math.isclose(stored_num, predicted_num, rel_tol=1e-2, abs_tol=1e-5)
    except Exception:
        return predicted.lower() == stored_answer.lower()


def exact_answer_match(predictions: list[str], answers: list[str]) -> dict[str, Any]:
    extracted_predictions = [extract_final_answer(str(pred)) for pred in predictions]
    total = len(answers)
    correct = sum(
        verify_answer(str(answer), str(pred))
        for pred, answer in zip(extracted_predictions, answers)
    )
    return {
        "correct": correct,
        "total": total,
        "accuracy": correct / total if total else 0.0,
        "extracted_predictions": extracted_predictions,
    }


def format_prompt(prompt: str) -> str:
    return (
        f"{prompt}\n"
        "Please put your final answer inside `\\boxed{}`. "
        "For example: `\\boxed{your answer}`\n"
    )


def format_training_text(prompt: str, answer: str) -> str:
    return f"{format_prompt(prompt)}\\boxed{{{answer}}}"


def extract_answer(generated_text: str, prompt_text: str | None = None) -> str:
    text = generated_text
    if prompt_text and text.startswith(prompt_text):
        text = text[len(prompt_text):]
    return extract_final_answer(text)


# def list_lora_target_candidates(model: nn.Module) -> list[tuple[str, str]]:
#     candidates: list[tuple[str, str]] = []
#     for name, module in model.named_modules():
#         lower = name.lower()
#         if isinstance(module, nn.Linear) and any(
#             key in lower for key in ["proj", "attn", "mlp", "moe", "expert", "gate"]
#         ):
#             candidates.append((name, repr(module)))
#     return candidates

In [4]:
# input: LoRA paramaters

from __future__ import annotations

from peft import TaskType

LORA_RANK = 4
LORA_ALPHA = 8
TARGET_MODULES = r".*\.(in_proj|out_proj|up_proj|down_proj)$"
LORA_DROPOUT = 0.05
LORA_BIAS = "none"
TASK_TYPE = TaskType.CAUSAL_LM

lora_component = LoRA_Comp(LORA_RANK,LORA_ALPHA,TARGET_MODULES,
                           LORA_DROPOUT,LORA_BIAS,TASK_TYPE)

In [2]:
# input: Data Selection
row_select=[37,72,144,152,216,288,290,360,397,469,516,541,613,722,798,829,831,901,1010,1137,1154,1160,1191,1263,1298,1370,1407,1409,1442,1514,1516,1551,1588,1592,1623,1625,1629,1695,1732,1767,1816,1820,1839,1847,1876,1911,1948,1983,2020,2055,2057,2059,2168,2236,2308,2382,2417,2452,2456,2524,2542,2567,2596,2598,2606,2633,2635,2645,2705,2742,2777,2849,2853,2861,2921,2993,2995,3017,3034,3065,3137,3176,3209,3219,3283,3320,3344,3369,3390,3439,3534,3536,3538,3575,3606,3643,3678,3715,3750,3787,3840,3861,3889,3931,3968,3970,4075,4077,4184,4202,4219,4258,4291,4365,4437,4472,4474,4516,4544,4581,4583,4688,4694,4696,4725,4727,4760,4762,4832,4904,4906,4910,4916,4988,5013,5033,5050,5085,5093,5130,5194,5231,5237,5278,5301,5338,5373,5375,5410,5445,5477,5484,5525,5530,5554,5698,5737,5770,5774,5809,5842,5846,5852,5879,5914,5988,6060,6066,6095,6099,6239,6276,6280,6420,6492,6527,6599,6681,6708,6710,6782,6788,6817,6854,6858,6924,6996,7010,7033,7068,7140,7142,7144,7214,7286,7321,7366,7393,7436,7506,7537,7609,7648,7681,7718,7790,7800,7899,7903,7936,8006,8045,8078,8125,8150,8222,8259,8331,8368,8403,8475,8547,8619,8658,8662,8691,8728,8763,8765,8800,8835,8837,8876,8909,8944,8950,9016,9053,9088,9100,9125,9160,9197,9232,9234,9269,9304,9341,9376,9487]

In [ ]:
# input: TRAINING_CONFIG
TRAIN_CSV = Path("/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv")
TEST_CSV = Path("/kaggle/input/nvidia-nemotron-3-reasoning-challenge/test.csv")

LOCAL_MODEL_PATH = Path(
    r"E:\\Projects\\Co_AMAP\\Co_Kaggle\\models\\huggingface\\hub\\models--TinyLlama--TinyLlama-1.1B-Chat-v1.0"
)
KAGGLE_MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")

# Use "local" for an 8 GB laptop workflow check.
# Use "kaggle" after switching MODEL_PATH to KAGGLE_MODEL_PATH.
RUN_MODE = "kaggle"
MODEL_PATH = LOCAL_MODEL_PATH if RUN_MODE == "local" else KAGGLE_MODEL_PATH

LOCAL_TRAINING_CONFIG = {
    "MAX_TRAIN_ROWS": 2,
    "MAX_VAL_ROWS": 2,
    "EPOCHS": 1,
    "MAX_STEPS": 1,
    "BATCH_SIZE": 1,
    "GRADIENT_ACCUMULATION_STEPS": 1,
    "LEARNING_RATE": 1e-4,
    "MAX_LENGTH": 128,  # max tokens for each training prompt+answer example
    "MAX_NEW_TOKENS": 16,  # max answer tokens generated during scoring
    "USE_EARLY_STOPPING": False,
    "EVAL_EVERY_STEPS": None,
    "EARLY_STOPPING_PATIENCE": None,
    "SAVE_ADAPTER": False,
    "ADAPTER_OUTPUT_DIR": Path("lora_adapter_LoRA_001"),
}



OUTPUT_DIR = "/kaggle/working"
KAGGLE_TRAINING_CONFIG = {
    "TRAIN_FRACTION": 0.80,
    "EPOCHS": 1,
    "MAX_STEPS": 200,  # hard stop: max number of training batches
    "BATCH_SIZE": 1,
    "GRADIENT_ACCUMULATION_STEPS": 8,
    "LEARNING_RATE": 1e-4,
    "MAX_LENGTH": 512,  # max tokens for each training prompt+answer example
    "MAX_NEW_TOKENS": 32,  # max answer tokens generated during scoring
    "USE_EARLY_STOPPING": True,
    "EVAL_EVERY_STEPS": 10,  # evaluate every N optimizer updates
    "EARLY_STOPPING_VAL_ROWS": 50,  # use a small validation subset for fast stopping checks
    "STOP_WHEN_VAL_WORSE": True,
    "SAVE_ADAPTER": True,
    "ADAPTER_OUTPUT_DIR": OUTPUT_DIR,
}

TRAINING_CONFIG = LOCAL_TRAINING_CONFIG if RUN_MODE == "local" else KAGGLE_TRAINING_CONFIG
TRAINING_CONFIG

NameError: name 'kagglehub' is not defined

In [ ]:
# data load and split

train_raw = pl.read_csv(TRAIN_CSV)[row_select] if len(row_select)>0 else pl.read_csv(TRAIN_CSV)
test_df = pl.read_csv(TEST_CSV)

if RUN_MODE == "local":
    train_df, val_df = make_train_val_split(
        train_raw,
        max_train_rows=TRAINING_CONFIG["MAX_TRAIN_ROWS"],
        max_val_rows=TRAINING_CONFIG["MAX_VAL_ROWS"],
    )
if RUN_MODE == "kaggle":
    split_idx = int(train_raw.height * TRAINING_CONFIG["TRAIN_FRACTION"])
    train_df = train_raw.slice(0, split_idx)
    val_df = train_raw.slice(split_idx)

train_rows = train_df.to_dicts()
val_rows = val_df.to_dicts()
val_answers = [str(row["answer"]) for row in val_rows]

print(f"train rows: {len(train_rows)}")
print(f"validation rows: {len(val_rows)}")
print(f"test rows: {test_df.height}")


NameError: name 'TRAIN_CSV' is not defined

In [7]:
#TRITON_PTXAS_BLACKWELL_PATH
import os
import shutil
from pathlib import Path

src = Path(
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/"
    "triton/backends/nvidia/bin/ptxas-blackwell"
)
dst = Path("/kaggle/working/ptxas-blackwell")

shutil.copy(src, dst)
os.chmod(dst, 0o755)

os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = str(dst)

In [8]:
#a class of funcs for load, train model

class PromptAnswerDataset(Dataset):
    """Builds tokenized examples with loss only on the answer completion.

    The model still receives prompt + `\\boxed{answer}` as input, but prompt
    token labels are set to -100. This aligns SFT loss with the competition
    metric, which scores only the extracted final answer.
    """

    def __init__(self, rows, tokenizer, max_length):
        self.rows = rows
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.rows)

    def _tokenize_no_specials(self, text):
        return self.tokenizer(text, add_special_tokens=False)["input_ids"]

    def __getitem__(self, idx):
        row = self.rows[idx]
        prompt_text = format_prompt(str(row["prompt"]))
        answer_text = f"\\boxed{{{row['answer']}}}"

        prompt_ids = self._tokenize_no_specials(prompt_text)
        answer_ids = self._tokenize_no_specials(answer_text)
        if self.tokenizer.eos_token_id is not None:
            answer_ids = answer_ids + [self.tokenizer.eos_token_id]

        answer_ids = answer_ids[: self.max_length]
        prompt_budget = max(self.max_length - len(answer_ids), 0)
        prompt_ids = prompt_ids[:prompt_budget]

        input_values = prompt_ids + answer_ids
        label_values = [-100] * len(prompt_ids) + answer_ids
        attention_values = [1] * len(input_values)

        pad_len = self.max_length - len(input_values)
        if pad_len > 0:
            input_values += [self.tokenizer.pad_token_id] * pad_len
            label_values += [-100] * pad_len
            attention_values += [0] * pad_len

        return {
            "input_ids": torch.tensor(input_values, dtype=torch.long),
            "attention_mask": torch.tensor(attention_values, dtype=torch.long),
            "labels": torch.tensor(label_values, dtype=torch.long),
        }


def load_tokenizer(model_dir):
    tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def load_model(model_dir):
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        trust_remote_code=True,
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
    )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return model.to(device)


def generate_predictions(model, tokenizer, rows, max_new_tokens):
    model.eval()
    predictions = []
    device = next(model.parameters()).device
    with torch.no_grad():
        for row in rows:
            prompt_text = format_prompt(str(row["prompt"]))
            encoded = tokenizer(prompt_text, return_tensors="pt").to(device)
            output_ids = model.generate(
                **encoded,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
            generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
            predictions.append(extract_answer(generated, prompt_text))
    return predictions


def evaluate_validation_score(model, tokenizer, rows, answers, config):
    predictions = generate_predictions(
        model,
        tokenizer,
        rows,
        max_new_tokens=config["MAX_NEW_TOKENS"],
    )
    return exact_answer_match(predictions, answers)


def train_supervised_lora(model, tokenizer, train_rows, val_rows, val_answers, config):
    dataset = PromptAnswerDataset(train_rows, tokenizer, config["MAX_LENGTH"])
    loader = DataLoader(dataset, batch_size=config["BATCH_SIZE"], shuffle=False)
    optimizer = torch.optim.AdamW(
        (param for param in model.parameters() if param.requires_grad),
        lr=config["LEARNING_RATE"],
    )
    device = next(model.parameters()).device
    steps = 0
    optimizer_updates = 0
    epochs_completed = 0
    last_loss = None
    best_val_score = None
    best_val_accuracy = -1.0
    early_stop_val_rows = val_rows[: config.get("EARLY_STOPPING_VAL_ROWS", len(val_rows))]
    early_stop_val_answers = val_answers[: len(early_stop_val_rows)]
    model.train()
    optimizer.zero_grad(set_to_none=True)

    for epoch_idx in range(config["EPOCHS"]):
        current_epoch = epoch_idx + 1
        print(f"epoch {current_epoch}/{config['EPOCHS']} started")
        for batch in loader:
            batch = {key: value.to(device) for key, value in batch.items()}
            loss = model(**batch).loss / config["GRADIENT_ACCUMULATION_STEPS"]
            loss.backward()
            last_loss = float(loss.detach().cpu() * config["GRADIENT_ACCUMULATION_STEPS"])

            if (steps + 1) % config["GRADIENT_ACCUMULATION_STEPS"] == 0:
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                optimizer_updates += 1

                if (
                    config["USE_EARLY_STOPPING"]
                    and config["EVAL_EVERY_STEPS"] is not None
                    and optimizer_updates % config["EVAL_EVERY_STEPS"] == 0
                ):
                    val_score = evaluate_validation_score(
                        model,
                        tokenizer,
                        early_stop_val_rows,
                        early_stop_val_answers,
                        config,
                    )
                    model.train()
                    print(f"validation at update {optimizer_updates}: {val_score}")

                    if val_score["accuracy"] > best_val_accuracy:
                        best_val_accuracy = val_score["accuracy"]
                        best_val_score = val_score
                    elif (
                        config.get("STOP_WHEN_VAL_WORSE", False)
                        and val_score["accuracy"] < best_val_accuracy
                    ):
                        return {
                            "steps": steps + 1,
                            "optimizer_updates": optimizer_updates,
                            "current_epoch": current_epoch,
                            "epochs_completed": epochs_completed,
                            "last_loss": last_loss,
                            "best_val_score": best_val_score,
                            "current_val_score": val_score,
                            "stopped_by": "validation_worse",
                        }

            steps += 1
            if config["MAX_STEPS"] is not None and steps >= config["MAX_STEPS"]:
                return {
                    "steps": steps,
                    "optimizer_updates": optimizer_updates,
                    "current_epoch": current_epoch,
                    "epochs_completed": epochs_completed,
                    "last_loss": last_loss,
                    "best_val_score": best_val_score,
                    "stopped_by": "max_steps",
                }

        epochs_completed += 1
        print(f"epoch {current_epoch}/{config['EPOCHS']} completed")

    return {
        "steps": steps,
        "optimizer_updates": optimizer_updates,
        "current_epoch": config["EPOCHS"],
        "epochs_completed": epochs_completed,
        "last_loss": last_loss,
        "best_val_score": best_val_score,
        "stopped_by": "epochs_complete",
    }

In [ ]:
# loading model weights and show some model info
model_dir = resolve_model_path(MODEL_PATH)
tokenizer = load_tokenizer(model_dir)
model = load_model(model_dir)

print(f"model dir: {model_dir}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

In [ ]:
#Baseline score before fine tuning, but just to check if model works.

baseline_predictions = generate_predictions(
    model,
    tokenizer,
    val_rows[:10],
    max_new_tokens=TRAINING_CONFIG["MAX_NEW_TOKENS"],
)
baseline_score = exact_answer_match(baseline_predictions, val_answers)
print("Baseline score:", baseline_score)
baseline_predictions

In [ ]:
# LoRA attach and Fine tuning train
model = attach_lora(model, lora_component)
model.print_trainable_parameters()

train_summary = train_supervised_lora(
    model,
    tokenizer,
    train_rows,
    val_rows,
    val_answers,
    TRAINING_CONFIG,
)
print("Training summary:", train_summary)

In [ ]:
# score after fine tuning, but just to check if model works.
after_training_predictions = generate_predictions(
    model,
    tokenizer,
    val_rows[:10],
    max_new_tokens=TRAINING_CONFIG["MAX_NEW_TOKENS"],
)
after_training_score = exact_answer_match(after_training_predictions, val_answers)
print("After-training score:", after_training_score)
after_training_predictions

In [ ]:
# Save Adapter
print(f"Saving adapter to {OUTPUT_DIR}...")
model.save_pretrained(OUTPUT_DIR)

In [ ]:
import subprocess

subprocess.run("zip -m submission.zip *", shell=True, check=True)

In [ ]:
print('Done.')